In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

In [3]:
sentences = [
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]
labels = [1]*15 + [0]*15
labels = np.array(labels)

In [6]:
vocab_size = 2000
tok = Tokenizer(num_words = vocab_size, oov_token = "<OOV>")
tok.fit_on_texts(sentences)
seqs = tok.texts_to_sequences(sentences)
maxlen = max(len(s) for s in seqs)
X = pad_sequences(seqs, maxlen = maxlen, padding = 'post' )
y = labels

In [10]:
embed_dim = 16
rnn_units = 64

In [13]:
inp = Input(shape = (maxlen,), dtype = "int32", name = 'input')
x = Embedding(input_dim = vocab_size, output_dim = embed_dim, mask_zero = True, name = 'embed')(inp)
rnn = SimpleRNN(units = rnn_units, return_sequences = False, name = 'simple_rnn')
x_last = rnn(x)
out = Dense(1, activation = 'sigmoid', name = 'out')(x_last)
model = Model(inputs = inp, outputs = out)
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

In [14]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │     32,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 5)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 64)        │      5,184 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │         65 │ simple_rnn[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 37,249 (145.50 KB)

 Trainable params: 37,249 (145.50 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.fit(X, y, epochs = 25, batch_size = 8, verbose = 1)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.4667 - loss: 0.6958
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6000 - loss: 0.6762
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7333 - loss: 0.6587
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8000 - loss: 0.6404
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8333 - loss: 0.6172
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.8667 - loss: 0.5865
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8667 - loss: 0.5514
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8667 - loss: 0.5036
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9000 - loss: 0.4473
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9000 - loss: 0.3927
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9333 - loss: 0.3297
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9333 - loss: 0.2723
E

In [17]:
intermediate_model = Model(inputs=model.inputs, outputs=[model.get_layer('embed').output, model.get_layer('simple_rnn').output])

In [18]:
from tensorflow.keras.layers import SimpleRNN as SRNN
seq_inp = Input(shape=(maxlen,), dtype='int32')
seq_emb = model.get_layer('embed')(seq_inp)  # reuse trained embedding

# Create RNN with return_sequences=True
rnn_seq = SRNN(units=rnn_units, return_sequences=True, name='rnn_seq')

# DO NOT CALL build() manually
seq_hidden = rnn_seq(seq_emb)  # builds automatically

# Copy trained RNN weights
try:
    trained_weights = model.get_layer('simple_rnn').get_weights()
    rnn_seq.set_weights(trained_weights)
    print("Copied RNN weights into sequence-inspection RNN.")
except Exception as e:
    print("Could not copy weights automatically:", e)

inspect_model = Model(inputs=seq_inp, outputs=seq_hidden)

# Inspect
idx = 0
example_seq = X[idx:idx+1]  # shape (1, maxlen)
hidden_seq = inspect_model.predict(example_seq)

print("Sentence:", sentences[idx])
print("Token ids:", example_seq)
print("Hidden states per timestep shape:", hidden_seq.shape)
print("Hidden states (timesteps x units):")
print(np.round(hidden_seq[0], 3))

Copied RNN weights into sequence-inspection RNN.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 222ms/step
Sentence: I love this product
Token ids: [[ 3 26  2  7  0]]
Hidden states per timestep shape: (1, 5, 64)
Hidden states (timesteps x units):
[[-0.014 -0.003  0.031  0.022  0.011 -0.02  -0.006  0.014 -0.01   0.019
   0.     0.043  0.005 -0.006 -0.032  0.019  0.02  -0.002 -0.02   0.022
  -0.047 -0.001  0.005  0.01   0.016 -0.003 -0.012  0.001 -0.001 -0.017
   0.011  0.034 -0.043  0.016 -0.004 -0.016 -0.001 -0.01   0.039  0.042
   0.003 -0.039 -0.021  0.019  0.013 -0.013 -0.056  0.013  0.008  0.009
  -0.006 -0.016  0.005  0.006 -0.001 -0.002  0.011  0.009  0.026 -0.016
  -0.018 -0.026  0.018  0.014]
 [-0.034 -0.041 -0.001  0.095  0.041  0.132 -0.064 -0.006 -0.049 -0.026
  -0.05   0.004  0.125 -0.055 -0.015  0.04  -0.047 -0.073  0.043  0.096
  -0.055 -0.082  0.052  0.026  0.017  0.131 -0.072 -0.019 -0.06   0.026
   0.126 -0.064 -0.055 -0.07  -0.022  0.042 -0.024 -0.072  0.077  0.041
   0.026 -0.049 -0.05